# 04 - Historical Training and Recency Research

Reuse or train the historical TCN+GRU control, then compare causal recency-aware ensemble policies on historical rolling-origin windows only. The failed June 2026 future-OOS window is never used for policy selection.

In [ ]:
import torch, sys
assert torch.cuda.is_available(), "Enable GPU: Runtime -> Change runtime -> T4/A100"
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_BASE  = '/content/drive/MyDrive/yeniBot'
DATA_DIR    = f'{DRIVE_BASE}/data'
CHECKPT_DIR = f'{DRIVE_BASE}/checkpoints'
os.makedirs(f'{DATA_DIR}/processed', exist_ok=True)
os.makedirs(CHECKPT_DIR, exist_ok=True)

In [ ]:
import os, subprocess, sys
REPO_URL = 'https://github.com/umutergul74/yeniBot.git'
REPO_DIR = '/content/yenibot_repo'
REPO_BRANCH = os.environ.get('YENIBOT_REPO_BRANCH', 'research/next-candidate-v1')
if os.path.exists(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', '-B', REPO_BRANCH, f'origin/{REPO_BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, REPO_DIR], check=True)
repo_commit = subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'], text=True).strip()
repo_branch = subprocess.check_output(['git', '-C', REPO_DIR, 'branch', '--show-current'], text=True).strip()
assert repo_branch == REPO_BRANCH, f'Expected {REPO_BRANCH}, found {repo_branch}'
sys.path.insert(0, REPO_DIR)
print('Repository branch:', repo_branch)
print('Repository commit:', repo_commit)
print('After a changed checkout, use Runtime -> Restart session before trusting imports.')

In [ ]:
!pip install -q -r {REPO_DIR}/requirements.txt

In [ ]:
import yaml
with open(f'{REPO_DIR}/config.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['paths']['data_dir'] = DATA_DIR
cfg['paths']['checkpoint_dir'] = CHECKPT_DIR
research = cfg.get('experiments', {}).get('next_research_cycle', {})
failed_outcomes = cfg.get('experiments', {}).get('frozen_candidate_outcomes', {})
assert research.get('status') == 'open_after_failed_future_oos'
assert research.get('same_window_selection_allowed') is False
assert research.get('new_future_oos_anchor_required') is True
print('Research cycle:', research.get('status'))
print('Primary hypothesis:', research.get('primary_hypothesis'))
print('Retired frozen candidates:', list(failed_outcomes))

In [ ]:
AUTO_UNASSIGN = True
RUN_RECENCY_RESEARCH = True

import json
import pandas as pd
from pathlib import Path
from google.colab import runtime
from yenibot.experiment import (
    experiment_settings,
    prepare_training_holdout_split,
    profile_run_dir,
    run_experiment_matrix,
    run_recency_ensemble_research,
)

labeled_path = f'{DATA_DIR}/processed/labeled_1h.parquet'
if not os.path.exists(labeled_path):
    raise FileNotFoundError(f'Missing labeled input; run notebooks 01-03 first: {labeled_path}')
labeled_full = pd.read_parquet(labeled_path)
assert labeled_full['timestamp'].is_monotonic_increasing
assert not labeled_full['timestamp'].duplicated().any()
holdout_path = f'{DATA_DIR}/processed/holdout_1h.parquet'
labeled, holdout, holdout_meta = prepare_training_holdout_split(
    labeled_full,
    cfg,
    holdout_path=holdout_path,
)
min_selection_rows = (
    cfg['walk_forward']['train_bars']
    + cfg['walk_forward']['val_bars']
    + cfg['walk_forward']['test_bars']
    + cfg['walk_forward']['purge_bars']
    + cfg['walk_forward']['embargo_bars']
)
if len(labeled) <= min_selection_rows:
    raise ValueError(
        f'Not enough selection rows for one full CV fold after holdout split: {len(labeled)} rows'
    )
cfg.setdefault('experiments', {})['holdout'] = holdout_meta

print('Full labeled rows:', len(labeled_full))
print('Selection/training rows:', len(labeled))
print('Holdout rows:', len(holdout))
print('Selection data end:', holdout_meta['selection_data_end'])
print('Holdout data start:', holdout_meta['holdout_data_start'])
print('Holdout split mode:', holdout_meta.get('split_mode'))
print('Unused rows after anchor:', holdout_meta.get('unused_rows_after_anchor'))
print('Future OOS ready:', holdout_meta.get('future_oos_ready'))
print('Future OOS next action:', holdout_meta.get('next_action'))
print('Holdout saved:', holdout_path)
settings = experiment_settings(cfg)
print('Experiment mode:', settings.get('mode', 'staged'))
print('Active feature profile:', cfg.get('features', {}).get('active_profile'))
print('Control profile:', settings.get('control_profile'))
print('Candidate profiles:', settings.get('candidate_profiles', []))
print('Skipped historical profiles:', settings.get('skipped_profiles', []))
print('Full CV mode:', settings.get('full_cv_profiles', 'auto'))
print('Always-full profiles:', settings.get('always_full_profiles', []))
print('Max auto full candidates:', settings.get('max_auto_full_candidates'))
print('Seed audit:', settings.get('seed_audit', {}))
print('Seed:', cfg['project']['random_seed'], 'deterministic:', cfg['project'].get('deterministic', False))

try:
    result = run_experiment_matrix(
        labeled,
        cfg,
        checkpoint_dir=CHECKPT_DIR,
    )
    print('Experiment run:', result['run_id'])
    print('Experiment dir:', result['run_dir'])
    print('Run id source:', result.get('run_id_source'))
    print('Training executed scopes:', result.get('training_executed_count'))
    print('Training reused scopes:', result.get('training_skipped_count'))
    print('All training scopes reused:', result.get('all_training_scopes_reused'))
    display(result['comparison'])
    if not result.get('missing_selected_profiles').empty:
        display(result['missing_selected_profiles'])
    if not result.get('seed_stability').empty:
        display(result['seed_stability'])
    if not result.get('seed_ensemble').empty:
        display(result['seed_ensemble'])
    if not result.get('experiment_policy_guard').empty:
        print('Experiment policy guard:')
        display(result['experiment_policy_guard'])
    if not result.get('future_oos_candidate_plan').empty:
        print('Future OOS candidate plan:')
        display(result['future_oos_candidate_plan'])
    if not result.get('performance_gap_analysis').empty:
        print('Performance gap analysis:')
        display(result['performance_gap_analysis'])
    if not result.get('payoff_alignment_summary').empty:
        print('Payoff alignment summary:')
        display(result['payoff_alignment_summary'])
    if not result.get('payoff_policy_robustness_summary').empty:
        print('Payoff policy robustness summary:')
        display(result['payoff_policy_robustness_summary'])
    recency_completed = False
    recency_cfg = cfg.get('experiments', {}).get('next_research_cycle', {}).get('recency_ensemble', {})
    if RUN_RECENCY_RESEARCH and recency_cfg.get('enabled', False):
        control_scope = profile_run_dir(
            CHECKPT_DIR,
            result['run_id'],
            settings['control_profile'],
        ) / 'full'
        recency_result = run_recency_ensemble_research(
            frame=labeled,
            scope_dir=control_scope,
            config=cfg,
            output_dir=result['run_dir'] / 'recency_research',
        )
        print('Recency ensemble research:', recency_result.get('status'))
        print('Recency research directory:', result['run_dir'] / 'recency_research')
        print('Fit operations in recency research: 0')
        print('Failed future OOS used for policy selection: False')
        recency_completed = recency_result.get('status') in {'completed', 'reused'}
        if not recency_result.get('summary', pd.DataFrame()).empty:
            display(recency_result['summary'])
        if not recency_result.get('paired_comparison', pd.DataFrame()).empty:
            print('Paired recency policy comparison:')
            display(recency_result['paired_comparison'])
        if recency_result.get('decision'):
            print('Recency policy decision:')
            print(json.dumps(recency_result['decision'], indent=2))
        audit = recency_result.get('eligibility_audit', pd.DataFrame())
        if not audit.empty:
            assert audit['eligible'].all(), 'Recency eligibility audit failed'
            print('Eligibility audit rows:', len(audit), '- all passed')
    handoff_path = Path(CHECKPT_DIR) / 'notebook04_run.json'
    handoff_path.write_text(json.dumps({
        'run_id': result['run_id'],
        'repo_branch': repo_branch,
        'repo_commit': repo_commit,
        'recency_research_completed': recency_completed,
    }, indent=2), encoding='utf-8')
    print('Diagnostics run handoff:', handoff_path)
    print('Decision:', result['decision'])
finally:
    if AUTO_UNASSIGN:
        runtime.unassign()


In [ ]:
# Runtime is released once, after training reuse and recency research finish.
# Outputs are profile-isolated under: {CHECKPT_DIR}/experiments/{run_id}/{profile}/{fold_scope}/
# Notebook 05 resolves the exact run from: {CHECKPT_DIR}/notebook04_run.json
